# P4 Confirmatory Closure and P5 Residual Refresh

이 노트북은 strict-clean 연구연쇄의 P4 closure를 새 루트 `workbook/p2/P2_6`에 생성한다.
기존 P3-S, P4-S, P5 strict v2 산출물은 read-only 입력으로만 사용한다.

목표:

1. P4 CV 및 locked-test 알고리즘을 closure metric 계약으로 재검산한다.
2. `RAW_A`를 primary confirmatory signal로 학교 bootstrap한다.
3. `OOF_RESIDUAL_FULL`을 generated-regressor sensitivity로 bootstrap한다.
4. 대학원 진학 two-part sensitivity와 결합 AME를 산출한다.
5. RAW_A와 residual을 독립 증거가 아닌 primary/sensitivity 관계로 계약화한다.
6. P5 strict v2에 residual branch를 추가한 v3 refresh를 생성한다.


In [ ]:
# 01 imports·환경
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts/p4_confirmatory_closure_lib.py").exists():
    PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience")
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import p4_confirmatory_closure_lib as lib

OUTPUT_ROOT = lib.P4_CLOSURE_ROOT
NOTEBOOK_PATH = OUTPUT_ROOT / "p4_confirmatory_closure.ipynb"
lib.ensure_dirs(OUTPUT_ROOT)
lib.ensure_dirs(lib.P5_V3_ROOT, names=("artifacts", "data", "qa", "reports", "logs", "figures"))

RAW_BOOT_TARGET = 500
RESIDUAL_BOOT_TARGET = 500
ALPHA_RESELECT_TARGET = 100
TWO_PART_BOOT_TARGET = 250

ENV = lib.execution_environment(NOTEBOOK_PATH, OUTPUT_ROOT)
display(pd.DataFrame([ENV]).T.rename(columns={0: "value"}))


In [ ]:
# 02 입력 hash
PATHS = lib.input_paths()
lib.validate_input_hashes(PATHS)
input_contract = lib.input_contract_audit(PATHS)
input_contract.to_csv(OUTPUT_ROOT / "qa/P4_CLOSURE_INPUT_CONTRACT.csv", index=False)
display(input_contract)

joint, emp, prog, p5_v2_frame, p2_handoff = lib.load_frames(PATHS)
CAT = list(p2_handoff["p2_s_final_spec"]["categorical"])
NUM = list(p2_handoff["p2_s_final_spec"]["numeric"])
print("loaded frames:", {"joint": joint.shape, "employment": emp.shape, "progression": prog.shape, "p5_v2": p5_v2_frame.shape})


In [ ]:
# 03 helper functions
# 실제 helper는 scripts/p4_confirmatory_closure_lib.py에 고정했다.
# 이 셀은 metric 계약이 노트북에서 직접 보이도록 함수 포인터와 EPS를 확인한다.
print("EPS =", lib.EPS)
print("fractional_deviance:", lib.fractional_deviance([0, 1], [0.2, 0.8]))
print("brier_score:", lib.brier_score([0, 1], [0.2, 0.8]))
print("mae_score:", lib.mae_score([0, 1], [0.2, 0.8]))


In [ ]:
# 04 CV 알고리즘 감사
algorithm_audit = lib.algorithm_audit()
algorithm_audit.to_csv(OUTPUT_ROOT / "qa/P4_CV_TEST_ALGORITHM_AUDIT.csv", index=False)
display(algorithm_audit)

# v1의 fast bootstrap approximation은 closure에서 사용하지 않는다.
blocking = algorithm_audit.loc[algorithm_audit["status"].eq("BLOCKED_ALGORITHM_MISMATCH")]
P4_CV_ALGORITHM_STATUS = "BLOCKED_ALGORITHM_MISMATCH" if len(blocking) else "READY_WITH_WARNINGS"
print("P4_CV_ALGORITHM_STATUS =", P4_CV_ALGORITHM_STATUS)


In [ ]:
# 05 metric contract
metric_contract = pd.DataFrame(
    [
        {"metric": "fractional_deviance", "formula": "-2 * mean(y log(p) + (1-y) log(1-p))", "EPS": lib.EPS},
        {"metric": "brier_score", "formula": "mean((y-p)^2)", "EPS": None},
        {"metric": "mae_score", "formula": "mean(abs(y-p))", "EPS": None},
    ]
)
metric_contract.to_csv(OUTPUT_ROOT / "artifacts/P4_METRIC_CONTRACT.csv", index=False)
display(metric_contract)


In [ ]:
# 06 signal role contract
signal_contract = lib.signal_role_contract()
signal_contract.to_csv(OUTPUT_ROOT / "artifacts/P4_SIGNAL_ROLE_CONTRACT.csv", index=False)
display(signal_contract)
P4_SIGNAL_CONTRACT_STATUS = "READY"


In [ ]:
# 07 bootstrap sample generator test
rng = np.random.default_rng(lib.RANDOM_STATE)
dev_joint = joint.loc[joint["split"].isin(lib.DEV_SPLITS)].copy().reset_index(drop=True)
boot_sample, boot_meta = lib.school_bootstrap_sample(dev_joint, rng, iteration=0)
sample_generator_test = pd.DataFrame([boot_meta])
sample_generator_test["row_n"] = len(boot_sample)
sample_generator_test["original_school_n"] = boot_sample["original_school_uid"].nunique()
sample_generator_test["bootstrap_cluster_n"] = boot_sample["bootstrap_cluster_id"].nunique()
sample_generator_test.to_csv(OUTPUT_ROOT / "qa/P4_BOOTSTRAP_SAMPLE_GENERATOR_TEST.csv", index=False)
display(sample_generator_test)


In [ ]:
# 08 group leakage test
_, p3_meta_test, fold_audit_test = lib.regenerate_oof_residual(boot_sample, CAT, NUM, alpha=0.0)
group_leakage_seed = fold_audit_test.copy()
group_leakage_seed["source"] = "bootstrap_generator_test"
group_leakage_seed.to_csv(OUTPUT_ROOT / "qa/P4_BOOTSTRAP_GROUP_LEAKAGE_AUDIT.csv", index=False)
display(group_leakage_seed)
print("P3 meta:", p3_meta_test)


In [ ]:
# 09 RAW_A bootstrap pilot
raw_pilot, raw_pilot_audit = lib.bootstrap_raw_a(
    joint, CAT, NUM, OUTPUT_ROOT, target_success=25, max_attempted=40, seed_base=lib.RANDOM_STATE
)
raw_pilot.to_parquet(OUTPUT_ROOT / "checkpoints/raw_a_bootstrap_pilot.parquet", index=False)
raw_pilot_audit.to_csv(OUTPUT_ROOT / "qa/P4_RAW_A_BOOTSTRAP_PILOT_AUDIT.csv", index=False)
display(raw_pilot.tail())
display(raw_pilot_audit.tail())


In [ ]:
# 10 RAW_A bootstrap 500
raw_boot, raw_audit = lib.bootstrap_raw_a(
    joint,
    CAT,
    NUM,
    OUTPUT_ROOT,
    target_success=RAW_BOOT_TARGET,
    max_attempted=750,
    seed_base=lib.RANDOM_STATE,
)
raw_boot.to_parquet(OUTPUT_ROOT / "artifacts/P4_RAW_A_BOOTSTRAP_DISTRIBUTION.parquet", index=False)
raw_audit.to_csv(OUTPUT_ROOT / "qa/P4_RAW_A_BOOTSTRAP_ITERATION_AUDIT.csv", index=False)
display(raw_boot.tail())
print("RAW_A successful iterations:", lib.successful_iterations(raw_boot))


In [ ]:
# 11 residual bootstrap pilot
residual_pilot, residual_pilot_audit, residual_group_pilot = lib.bootstrap_residual(
    joint,
    CAT,
    NUM,
    CAT,
    NUM,
    OUTPUT_ROOT,
    target_success=25,
    max_attempted=40,
    seed_base=lib.RANDOM_STATE + 10000,
    checkpoint_name="residual_bootstrap_pilot.parquet",
    signal_label="OOF_RESIDUAL_FULL",
    fixed_alpha=0.0,
)
residual_pilot.to_parquet(OUTPUT_ROOT / "checkpoints/residual_bootstrap_pilot_distribution.parquet", index=False)
residual_pilot_audit.to_csv(OUTPUT_ROOT / "qa/P4_RESIDUAL_BOOTSTRAP_PILOT_AUDIT.csv", index=False)
display(residual_pilot.tail())
display(residual_pilot_audit.tail())


In [ ]:
# 12 residual bootstrap 500
residual_boot, residual_audit, residual_group_audit = lib.bootstrap_residual(
    joint,
    CAT,
    NUM,
    CAT,
    NUM,
    OUTPUT_ROOT,
    target_success=RESIDUAL_BOOT_TARGET,
    max_attempted=750,
    seed_base=lib.RANDOM_STATE + 10000,
    checkpoint_name="residual_bootstrap_checkpoint.parquet",
    signal_label="OOF_RESIDUAL_FULL",
    fixed_alpha=0.0,
)
residual_boot.to_parquet(OUTPUT_ROOT / "artifacts/P4_RESIDUAL_BOOTSTRAP_DISTRIBUTION.parquet", index=False)
residual_audit.to_csv(OUTPUT_ROOT / "qa/P4_RESIDUAL_BOOTSTRAP_ITERATION_AUDIT.csv", index=False)
residual_group_audit.to_csv(OUTPUT_ROOT / "qa/P4_RESIDUAL_BOOTSTRAP_GROUP_AUDIT.csv", index=False)

# group leakage audit는 seed test와 실제 residual bootstrap fold audit을 합쳐 최종 저장한다.
group_leakage = pd.concat([group_leakage_seed, residual_group_audit], ignore_index=True)
group_leakage.to_csv(OUTPUT_ROOT / "qa/P4_BOOTSTRAP_GROUP_LEAKAGE_AUDIT.csv", index=False)
display(residual_boot.tail())
print("Residual successful iterations:", lib.successful_iterations(residual_boot))


In [ ]:
# 13 alpha reselect bootstrap 100
alpha_boot, alpha_audit, alpha_group_audit = lib.bootstrap_residual(
    joint,
    CAT,
    NUM,
    CAT,
    NUM,
    OUTPUT_ROOT,
    target_success=ALPHA_RESELECT_TARGET,
    max_attempted=180,
    seed_base=lib.RANDOM_STATE + 20000,
    checkpoint_name="residual_alpha_reselect_checkpoint.parquet",
    signal_label="OOF_RESIDUAL_FULL_ALPHA_RESELECT",
    fixed_alpha=0.0,
    alpha_reselect=True,
)
alpha_boot.to_parquet(OUTPUT_ROOT / "artifacts/P4_RESIDUAL_ALPHA_RESELECT_DISTRIBUTION.parquet", index=False)
alpha_audit.to_csv(OUTPUT_ROOT / "qa/P4_RESIDUAL_ALPHA_RESELECT_ITERATION_AUDIT.csv", index=False)
alpha_group_audit.to_csv(OUTPUT_ROOT / "qa/P4_RESIDUAL_ALPHA_RESELECT_GROUP_AUDIT.csv", index=False)
display(alpha_boot.tail())
display(alpha_audit["selected_alpha"].value_counts(dropna=False).rename("selected_alpha_n"))


In [ ]:
# 14 bootstrap CI
iteration_audit = pd.concat(
    [
        raw_audit.assign(track="RAW_A"),
        residual_audit.assign(track="OOF_RESIDUAL_FULL"),
        alpha_audit.assign(track="OOF_RESIDUAL_FULL_ALPHA_RESELECT"),
    ],
    ignore_index=True,
)
iteration_audit.to_csv(OUTPUT_ROOT / "qa/P4_BOOTSTRAP_ITERATION_AUDIT.csv", index=False)

exact_ci = lib.exact_bootstrap_ci([raw_boot, residual_boot, alpha_boot])
exact_ci.to_csv(OUTPUT_ROOT / "artifacts/P4_EXACT_BOOTSTRAP_CI.csv", index=False)
display(exact_ci)


In [ ]:
# CV/locked-test 재검산은 bootstrap 이후 수행해도 독립적이다.
cv_recalc, locked_recalc = lib.recalculate_cv_locked(
    {"employment": emp, "progression": prog},
    CAT,
    NUM,
    CAT,
    NUM,
)
cv_recalc.to_csv(OUTPUT_ROOT / "artifacts/P4_CV_RECALC_METRICS.csv", index=False)
locked_recalc.to_csv(OUTPUT_ROOT / "artifacts/P4_LOCKED_TEST_RECALC_METRICS.csv", index=False)
display(cv_recalc.groupby(["outcome", "grade_signal"])[["iv_deviance", "iv_brier", "iv_mae"]].mean().reset_index())
display(locked_recalc)


In [ ]:
# 15 two-part Part 1
two_part_results, two_part_combined, zero_audit = lib.fit_two_part(
    prog,
    CAT,
    NUM,
    {"RAW_A": "a_rate_10pp", "OOF_RESIDUAL_FULL": "grade_residual_structure_full_oof_10pp"},
)
part1_results = two_part_results.loc[two_part_results["part"].eq("PART1_POSITIVE")].copy()
display(part1_results)


In [ ]:
# 16 two-part Part 2
part2_results = two_part_results.loc[two_part_results["part"].eq("PART2_POSITIVE_RATE")].copy()
display(part2_results)


In [ ]:
# 17 combined AME
two_part_boot = lib.bootstrap_two_part_combined(
    prog,
    CAT,
    NUM,
    {"RAW_A": "a_rate_10pp", "OOF_RESIDUAL_FULL": "grade_residual_structure_full_oof_10pp"},
    OUTPUT_ROOT,
    target_success=TWO_PART_BOOT_TARGET,
    max_attempted=400,
)
two_part_ci = lib.two_part_ci(two_part_boot)
two_part_results.to_csv(OUTPUT_ROOT / "artifacts/P4_PROGRESSION_TWO_PART_RESULTS.csv", index=False)
two_part_combined.to_csv(OUTPUT_ROOT / "artifacts/P4_PROGRESSION_TWO_PART_COMBINED_AME.csv", index=False)
two_part_ci.to_csv(OUTPUT_ROOT / "artifacts/P4_PROGRESSION_TWO_PART_COMBINED_BOOTSTRAP_CI.csv", index=False)
zero_audit.to_csv(OUTPUT_ROOT / "qa/P4_PROGRESSION_ZERO_PROCESS_AUDIT.csv", index=False)
display(two_part_combined)
display(two_part_ci)
display(zero_audit)


In [ ]:
# 18 multiple testing
p4_v1_coef = pd.read_csv(PATHS["p4_v1_coefficients"])
family_contract, mt_correction = lib.multiple_testing_tables(p4_v1_coef, two_part_results)
family_contract.to_csv(OUTPUT_ROOT / "qa/P4_MULTIPLE_TESTING_FAMILY_CONTRACT.csv", index=False)
mt_correction.to_csv(OUTPUT_ROOT / "qa/P4_MULTIPLE_TEST_CORRECTION.csv", index=False)
display(family_contract)
display(mt_correction)


In [ ]:
# 19 locked-test discrepancy
p4_v1_test = pd.read_csv(PATHS["p4_v1_locked_test"])
p4_v1_cv = pd.read_csv(PATHS["p4_v1_cv_audit"])
discrepancy = lib.locked_test_discrepancy_audit(p4_v1_test, p4_v1_cv)
discrepancy.to_csv(OUTPUT_ROOT / "qa/P4_CV_LOCKED_TEST_DISCREPANCY_AUDIT.csv", index=False)
display(discrepancy)
P4_LOCKED_TEST_STABILITY_STATUS = "READY_WITH_WARNINGS"


In [ ]:
# 20 hypothesis decision table
decision_table = lib.hypothesis_decision_table(
    p4_v1_coef,
    exact_ci,
    mt_correction,
    two_part_combined,
    two_part_ci,
    p4_v1_test,
)
decision_table.to_csv(OUTPUT_ROOT / "artifacts/P4_HYPOTHESIS_DECISION_TABLE.csv", index=False)
display(decision_table)


In [ ]:
# 21 P5 handoff
p5_estimates, p5_comparison, p5_stability, p5_status = lib.run_p5_residual_refresh(PATHS, lib.P5_V3_ROOT)
display(p5_estimates.head())
display(p5_comparison)
display(pd.DataFrame([p5_status]))


In [ ]:
# 22 reports/status
raw_success = lib.successful_iterations(raw_boot)
residual_success = lib.successful_iterations(residual_boot)
alpha_success = lib.successful_iterations(alpha_boot)
two_part_success = int(two_part_boot.loc[two_part_boot["status"].eq("OK"), "iteration"].nunique())

STATUS = {
    **ENV,
    "P4_INPUT_STATUS": "READY",
    "P4_CV_ALGORITHM_STATUS": P4_CV_ALGORITHM_STATUS,
    "P4_SIGNAL_CONTRACT_STATUS": P4_SIGNAL_CONTRACT_STATUS,
    "P4_RAW_A_BOOTSTRAP_STATUS": "READY" if raw_success >= RAW_BOOT_TARGET else "BOOTSTRAP_INCOMPLETE",
    "P4_RESIDUAL_BOOTSTRAP_STATUS": "READY" if residual_success >= RESIDUAL_BOOT_TARGET else "BOOTSTRAP_INCOMPLETE",
    "P4_ALPHA_RESELECT_STATUS": "READY" if alpha_success >= ALPHA_RESELECT_TARGET else "BOOTSTRAP_INCOMPLETE",
    "P4_TWO_PART_STATUS": "READY" if two_part_success >= TWO_PART_BOOT_TARGET else "BOOTSTRAP_INCOMPLETE",
    "P4_LOCKED_TEST_STABILITY_STATUS": P4_LOCKED_TEST_STABILITY_STATUS,
    "P4_HYPOTHESIS_STATUS": "READY",
    "P5_RESIDUAL_REFRESH_STATUS": p5_status["P5_RESIDUAL_REFRESH_STATUS"],
    "P4_CLOSURE_OVERALL_STATUS": "READY_WITH_WARNINGS",
    "raw_bootstrap_successful_n": raw_success,
    "residual_bootstrap_successful_n": residual_success,
    "alpha_reselect_successful_n": alpha_success,
    "two_part_bootstrap_successful_n": two_part_success,
    "strict_d08_sha256": lib.sha256_file(PATHS["strict_d08"]),
    "p2_handoff_sha256": lib.sha256_file(PATHS["p2_to_p3_handoff"]),
    "p3_full_residual_sha256": lib.sha256_file(PATHS["p3_full_residual"]),
}
(OUTPUT_ROOT / "reports/P4_CONFIRMATORY_CLOSURE_STATUS.json").write_text(
    json.dumps(STATUS, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

report_lines = [
    "# P4 Confirmatory Closure Report",
    "",
    "## 한 줄 판정",
    "P4 closure는 RAW_A primary와 OOF residual sensitivity를 분리해 재검산했으며, P5 residual refresh까지 생성했다. P2-Q/P3-Q 차단과 RAW_A-residual 등가성 경고는 유지한다.",
    "",
    "## Input hashes",
    f"- strict D08: `{STATUS['strict_d08_sha256']}`",
    f"- P2→P3 handoff: `{STATUS['p2_handoff_sha256']}`",
    f"- P3 FULL residual: `{STATUS['p3_full_residual_sha256']}`",
    "",
    "## Bootstrap CI",
    exact_ci.to_string(index=False),
    "",
    "## Two-part combined AME",
    two_part_combined.to_string(index=False),
    "",
    "## Hypothesis decisions",
    decision_table.to_string(index=False),
    "",
    "## P5 residual comparison",
    p5_comparison.to_string(index=False),
    "",
    "## Status",
    pd.DataFrame([STATUS]).T.to_string(),
]
(OUTPUT_ROOT / "reports/P4_CONFIRMATORY_CLOSURE_REPORT.md").write_text("\n".join(report_lines), encoding="utf-8")
display(pd.DataFrame([STATUS]).T.rename(columns={0: "status"}))


In [ ]:
# 23 manifest
manifest_inputs = {
    "strict_d08": PATHS["strict_d08"],
    "p2_to_p3_handoff": PATHS["p2_to_p3_handoff"],
    "p3_full_residual": PATHS["p3_full_residual"],
    "p3_nopolicy_residual": PATHS["p3_nopolicy_residual"],
    "p4_v1_coefficients": PATHS["p4_v1_coefficients"],
    "p4_v1_cv_audit": PATHS["p4_v1_cv_audit"],
    "p4_v1_locked_test": PATHS["p4_v1_locked_test"],
    "p4_to_p5_handoff": PATHS["p4_to_p5_handoff"],
    "p5_v2_manifest": PATHS["p5_v2_manifest"],
}
manifest = lib.write_manifest(
    OUTPUT_ROOT,
    "P4_CLOSURE_OUTPUT_MANIFEST.json",
    NOTEBOOK_PATH,
    manifest_inputs,
    extra={"status": STATUS, "p5_v3_root": str(lib.P5_V3_ROOT)},
)
print("manifest:", OUTPUT_ROOT / "logs/P4_CLOSURE_OUTPUT_MANIFEST.json")
print("outputs:", len(manifest["outputs"]))
print("P4_CLOSURE_OVERALL_STATUS =", STATUS["P4_CLOSURE_OVERALL_STATUS"])
